In [ ]:
%pip install "numpy<2.0.0" -q
%pip install reality-stone==0.2.5 transformers datasets accelerate -q


In [ ]:
import reality_stone as rs
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
TARGET_RANK = 256
NUM_LAYERS = 8
NUM_EPOCHS = 3
BATCH_SIZE = 2
MAX_LENGTH = 512
DATASET_SIZE = 5000

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Model: {MODEL_NAME}")


In [ ]:
from reality_stone import RSULFLayerCUDA, RSULFLMHeadCUDA

def extract_weights(model, layer_idx):
    layer = model.model.layers[layer_idx]
    return {
        'WQ': layer.self_attn.q_proj.weight.detach().float(),
        'WK': layer.self_attn.k_proj.weight.detach().float(),
        'W1': layer.mlp.gate_proj.weight.detach().float(),
        'W2': layer.mlp.down_proj.weight.detach().float(),
    }

rsulf_layers = []
for i in range(NUM_LAYERS):
    w = extract_weights(model, i)
    rsulf = RSULFLayerCUDA(
        wq=w['WQ'], wk=w['WK'],
        w1=w['W1'], w2=w['W2'],
        d_model=w['WQ'].shape[1], r=TARGET_RANK,
        device=device,
    )
    rsulf_layers.append(rsulf)
    comp, orig, ratio = rsulf.param_count()
    print(f"Layer {i}: {ratio:.2f}x compression")

print(f"\n{NUM_LAYERS} RS-ULF layers (CUDA) created")


In [ ]:
hidden_size = model.config.hidden_size
vocab_size = model.config.vocab_size
rsulf_head = RSULFLMHeadCUDA(rsulf_layers, hidden_size, vocab_size, device=device)
print(f"LM Head (CUDA): hidden={hidden_size}, vocab={vocab_size}")


In [ ]:
dataset = load_dataset("wikimedia/wikipedia", "20231101.ko", split=f"train[:{DATASET_SIZE}]")

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
tokenized.set_format("torch")
dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)
print(f"Dataset: {len(tokenized)} samples")


In [ ]:
for param in model.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(rsulf_head.lm_head.parameters(), lr=5e-5)

model.eval()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
        
        logits = rsulf_head(hidden_states.float())
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        
        loss = F.cross_entropy(shift_logits.view(-1, vocab_size), shift_labels.view(-1), ignore_index=tokenizer.pad_token_id)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    print(f"Epoch {epoch+1} - Avg Loss: {total_loss / len(dataloader):.4f}")


In [ ]:
save_path = "rsulf_finetuned.pt"
torch.save({
    "lm_head": rsulf_head.lm_head.state_dict(),
    "config": {
        "model_name": MODEL_NAME,
        "target_rank": TARGET_RANK,
        "num_layers": NUM_LAYERS,
        "hidden_size": hidden_size,
        "vocab_size": vocab_size,
    },
}, save_path)
print(f"Saved: {save_path}")


In [ ]:
from google.colab import files
files.download(save_path)
